In [13]:
from google.colab import files
files.download("books_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
# Summary statistics
print("DATASET SUMMARY")
print("=" * 35)

print("Total books:", len(df))
print(f"Average price: £{df['Price_numeric'].mean():.2f}")
print(f"Lowest price: £{df['Price_numeric'].min():.2f}")
print(f"Highest price: £{df['Price_numeric'].max():.2f}")

print("\nBooks by rating:")
print(df["Rating"].value_counts())

print("\nFive cheapest books:")
display(
    df.sort_values("Price_numeric")[
        ["Title", "Price", "Rating"]
    ].head(5)
)

DATASET SUMMARY
Total books: 100
Average price: £34.56
Lowest price: £10.16
Highest price: £58.11

Books by rating:
Rating
Three    22
One      22
Five     19
Two      19
Four     18
Name: count, dtype: int64

Five cheapest books:


,Title,Price,Rating
84,Patience,£10.16,Three
20,In Her Wake,£12.84,One
81,Princess Between Worlds (Wide-Awake Princess #5),£13.34,Five
80,"Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Pr...",£13.61,Five
10,"Starving Hearts (Triangular Trade Trilogy, #1)",£13.99,Two


In [11]:
from google.colab import files

files.download("books_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup
from urllib.parse import urljoin

base_url = "https://books.toscrape.com/"
next_url = urljoin(base_url, "catalogue/page-1.html")

all_books = []
max_pages = 5

for page in range(1, max_pages + 1):
    print(f"Scraping page {page}...")

    response = requests.get(next_url, timeout=20)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")
    listings = soup.select("article.product_pod")

    if not listings:
        print("No book listings found. Stopping.")
        break

    for book in listings:
        title_tag = book.select_one("h3 a")
        price_tag = book.select_one(".price_color")
        rating_tag = book.select_one(".star-rating")
        availability_tag = book.select_one(".availability")

        if title_tag is None or price_tag is None:
            continue

        rating_classes = rating_tag.get("class", []) if rating_tag else []
        rating = next(
            (c for c in rating_classes if c != "star-rating"),
            "Unknown"
        )

        price = price_tag.get_text(strip=True).replace("Â£", "£")

        all_books.append({
            "Title": title_tag.get("title", title_tag.get_text(strip=True)),
            "Price": price,
            "Rating": rating,
            "Availability": (
                availability_tag.get_text(" ", strip=True)
                if availability_tag else "Unknown"
            ),
            "Book_URL": urljoin(next_url, title_tag.get("href", ""))
        })

    next_link = soup.select_one("li.next a")

    if not next_link:
        break

    next_url = urljoin(next_url, next_link["href"])
    time.sleep(1)

df = pd.DataFrame(all_books)
df = df.drop_duplicates(subset=["Title"]).reset_index(drop=True)

df["Price_numeric"] = pd.to_numeric(
    df["Price"].str.replace("£", "", regex=False),
    errors="coerce"
)

df.to_csv("books_dataset.csv", index=False, encoding="utf-8-sig")

print("\nScraping completed!")
print("Total unique books:", len(df))
print("Missing values per column:")
print(df.isnull().sum())
print("\nFirst five records:")
display(df.head())

Scraping page 1...
Scraping page 2...
Scraping page 3...
Scraping page 4...
Scraping page 5...

Scraping completed!
Total unique books: 100
Missing values per column:
Title            0
Price            0
Rating           0
Availability     0
Book_URL         0
Price_numeric    0
dtype: int64

First five records:


,Title,Price,Rating,Availability,Book_URL,Price_numeric
0,A Light in the Attic,£51.77,Three,In stock,https://books.toscrape.com/catalogue/a-light-i...,51.77
1,Tipping the Velvet,£53.74,One,In stock,https://books.toscrape.com/catalogue/tipping-t...,53.74
2,Soumission,£50.10,One,In stock,https://books.toscrape.com/catalogue/soumissio...,50.10
3,Sharp Objects,£47.82,Four,In stock,https://books.toscrape.com/catalogue/sharp-obj...,47.82
4,Sapiens: A Brief History of Humankind,£54.23,Five,In stock,https://books.toscrape.com/catalogue/sapiens-a...,54.23


In [8]:
import pandas as pd
from urllib.parse import urljoin

books_data = []

for book in book_list:
    title = book.select_one("h3 a")["title"]

    price = book.select_one(".price_color").get_text(strip=True)
    price = price.replace("Â£", "£")

    availability = book.select_one(
        ".availability"
    ).get_text(" ", strip=True)

    rating_tag = book.select_one(".star-rating")
    rating = rating_tag.get("class", ["", "Unknown"])[1]

    link = book.select_one("h3 a")["href"]
    book_url = urljoin(url, link)

    books_data.append({
        "Title": title,
        "Price": price,
        "Rating": rating,
        "Availability": availability,
        "Book_URL": book_url
    })

df = pd.DataFrame(books_data)

df.to_csv("books_dataset.csv", index=False, encoding="utf-8-sig")

print("Dataset created successfully!")
print("Total records:", len(df))
print("\nDataset preview:")
display(df.head())

Dataset created successfully!
Total records: 20

Dataset preview:


,Title,Price,Rating,Availability,Book_URL
0,A Light in the Attic,£51.77,Three,In stock,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,£53.74,One,In stock,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,£50.10,One,In stock,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,£47.82,Four,In stock,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,£54.23,Five,In stock,https://books.toscrape.com/catalogue/sapiens-a...


In [7]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, "html.parser")

book_list = soup.select("article.product_pod")

print("Total books found:", len(book_list))
print("\nFirst 5 books:\n")

for book in book_list[:5]:
    title = book.select_one("h3 a")["title"]
    price = book.select_one(".price_color").get_text(strip=True)
    availability = book.select_one(".availability").get_text(strip=True)

    rating_tag = book.select_one(".star-rating")
    rating = rating_tag.get("class")[1]

    print("Title:", title)
    print("Price:", price)
    print("Rating:", rating)
    print("Availability:", availability)
    print("-" * 40)

Total books found: 20

First 5 books:

Title: A Light in the Attic
Price: Â£51.77
Rating: Three
Availability: In stock
----------------------------------------
Title: Tipping the Velvet
Price: Â£53.74
Rating: One
Availability: In stock
----------------------------------------
Title: Soumission
Price: Â£50.10
Rating: One
Availability: In stock
----------------------------------------
Title: Sharp Objects
Price: Â£47.82
Rating: Four
Availability: In stock
----------------------------------------
Title: Sapiens: A Brief History of Humankind
Price: Â£54.23
Rating: Five
Availability: In stock
----------------------------------------


In [6]:
import requests
url = "https://books.toscrape.com/"
response = requests.get(url, timeout=20)

print("Status code:", response.status_code)
print("website connected successfully!")
print("HTML length:", len(response.text))

Status code: 200
website connected successfully!
HTML length: 51294


In [5]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

print("All libraries are working!")
print("Request version:", requests.__version__)
print("Pandas version:", pd.__version__)

All libraries are working!
Request version: 2.32.4
Pandas version: 2.2.3


In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [3]:
import sys
!{sys.executable} -m pip install requests beautifulsoup4 pandas